# Proyecto de visión computacional con YOLO

Notebook completo y ordenado para comprobar CUDA, ejecutar detección sobre una imagen, estructurar los resultados, procesar un video sin agotar la RAM y comparar detección con tracking.

> Ejecuta las celdas en orden. Si reinicias el kernel, vuelve a comenzar desde la celda 1.

## 1. Configuración del entorno

In [ ]:
from pathlib import Path
import gc
import urllib.request

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
from IPython.display import Video, display
from ultralytics import YOLO

DISPOSITIVO = 0 if torch.cuda.is_available() else 'cpu'
CARPETA_DATOS = Path('datos')
CARPETA_RESULTADOS = Path('resultados')
CARPETA_DATOS.mkdir(exist_ok=True)
CARPETA_RESULTADOS.mkdir(exist_ok=True)

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
print('CUDA de PyTorch:', torch.version.cuda)
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Ultralytics:', ultralytics.__version__)
print('OpenCV:', cv2.__version__)

## 2. Carga del modelo

Se usa `yolo11n.pt`, la variante pequeña de YOLO 11, adecuada para una GTX 1050 Ti de 4 GB.

In [ ]:
modelo = YOLO('yolo11n.pt')
print('Modelo cargado correctamente')

## 3. Descarga validada de la imagen de prueba

In [ ]:
ruta_imagen = CARPETA_DATOS / 'bus.jpg'
url_imagen = 'https://' + 'ultralytics.com/images/bus.jpg'

if not ruta_imagen.exists():
    solicitud = urllib.request.Request(url_imagen, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(solicitud, timeout=60) as respuesta:
        contenido = respuesta.read()
    if len(contenido) < 1_000:
        raise RuntimeError('La descarga de la imagen no parece válida')
    ruta_imagen.write_bytes(contenido)

imagen_prueba = cv2.imread(str(ruta_imagen))
if imagen_prueba is None:
    raise RuntimeError(f'OpenCV no pudo abrir {ruta_imagen}')

print('Imagen:', ruta_imagen.resolve())
print('Forma:', imagen_prueba.shape)

## 4. Detección sobre una imagen

In [ ]:
resultados_imagen = modelo.predict(
    source=str(ruta_imagen),
    device=DISPOSITIVO,
    conf=0.25,
    imgsz=640,
    verbose=False
)
resultado_imagen = resultados_imagen[0]
print('Objetos detectados:', len(resultado_imagen.boxes))

In [ ]:
imagen_anotada = resultado_imagen.plot()
imagen_rgb = cv2.cvtColor(imagen_anotada, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(imagen_rgb)
plt.axis('off')
plt.title('Detección de objetos con YOLO 11')
plt.show()

## 5. Extracción de detecciones a una tabla

Cada fila representa un objeto. `x1, y1` son la esquina superior izquierda y `x2, y2` la esquina inferior derecha.

In [ ]:
detecciones = []

for numero, caja in enumerate(resultado_imagen.boxes, start=1):
    clase_id = int(caja.cls.item())
    confianza = float(caja.conf.item())
    x1, y1, x2, y2 = caja.xyxy[0].cpu().tolist()

    detecciones.append({
        'deteccion': numero,
        'clase_id': clase_id,
        'clase': resultado_imagen.names[clase_id],
        'confianza': confianza,
        'x1': round(x1, 1),
        'y1': round(y1, 1),
        'x2': round(x2, 1),
        'y2': round(y2, 1),
        'ancho': round(x2 - x1, 1),
        'alto': round(y2 - y1, 1),
    })

df_detecciones = pd.DataFrame(detecciones)
df_detecciones

In [ ]:
resumen_clases = (
    df_detecciones.groupby('clase')
    .agg(
        cantidad=('clase', 'size'),
        confianza_promedio=('confianza', 'mean'),
        confianza_minima=('confianza', 'min'),
        confianza_maxima=('confianza', 'max'),
    )
    .sort_values('cantidad', ascending=False)
    .reset_index()
)

columnas_confianza = ['confianza_promedio', 'confianza_minima', 'confianza_maxima']
resumen_clases[columnas_confianza] = (resumen_clases[columnas_confianza] * 100).round(2)
resumen_clases

## 6. Efecto del umbral de confianza

In [ ]:
for umbral in [0.25, 0.50, 0.70, 0.90]:
    filtradas = df_detecciones[df_detecciones['confianza'] >= umbral]
    conteo = filtradas['clase'].value_counts().to_dict()
    print(f'Umbral {umbral:.0%}: {len(filtradas)} detecciones ({conteo})')

## 7. Descarga e inspección del video

Se utiliza `vtest.avi` del repositorio oficial de OpenCV. La descarga se valida antes de continuar.

In [ ]:
ruta_video = CARPETA_DATOS / 'vtest.avi'
url_video = (
    'https://'
    + 'raw.githubusercontent.com/opencv/opencv/master/'
    + 'samples/data/vtest.avi'
)

if not ruta_video.exists():
    print('Descargando video...')
    solicitud = urllib.request.Request(url_video, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(solicitud, timeout=120) as respuesta:
        with ruta_video.open('wb') as archivo:
            while True:
                bloque = respuesta.read(1024 * 1024)
                if not bloque:
                    break
                archivo.write(bloque)

if ruta_video.stat().st_size < 1_000_000:
    raise RuntimeError('El archivo de video está incompleto o no es válido')

captura = cv2.VideoCapture(str(ruta_video))
if not captura.isOpened():
    raise RuntimeError(f'OpenCV no pudo abrir {ruta_video}')

fps = captura.get(cv2.CAP_PROP_FPS)
total_frames = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
ancho = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
captura.release()
duracion = total_frames / fps if fps else 0

print('Video:', ruta_video.resolve())
print(f'Tamaño: {ruta_video.stat().st_size / 1024**2:.2f} MB')
print(f'Resolución: {ancho}x{alto}')
print(f'FPS: {fps:.2f}')
print(f'Fotogramas: {total_frames}')
print(f'Duración: {duracion:.2f} segundos')

## 8. Detección de personas en video sin acumular RAM

`stream=True` entrega un resultado por vez. No se guarda una lista con todos los fotogramas en memoria.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

generador_deteccion = modelo.predict(
    source=str(ruta_video),
    device=DISPOSITIVO,
    conf=0.40,
    classes=[0],
    imgsz=480,
    vid_stride=2,
    stream=True,
    save=True,
    project=str(CARPETA_RESULTADOS),
    name='deteccion_video',
    exist_ok=True,
    verbose=False
)

fotogramas_deteccion = 0
carpeta_deteccion = None

for resultado_frame in generador_deteccion:
    fotogramas_deteccion += 1
    carpeta_deteccion = Path(resultado_frame.save_dir)
    if fotogramas_deteccion % 100 == 0:
        print(f'Procesados: {fotogramas_deteccion} fotogramas')

print('Detección terminada')
print('Fotogramas analizados:', fotogramas_deteccion)
print('Carpeta de salida:', carpeta_deteccion.resolve())

In [ ]:
archivos_deteccion = sorted(p for p in carpeta_deteccion.glob('*') if p.is_file())
for archivo in archivos_deteccion:
    print(f'{archivo.name}: {archivo.stat().st_size / 1024**2:.2f} MB')

## 9. Tracking: identidad persistente por objeto

La detección reconoce personas en cada fotograma. El tracking agrega un ID para intentar mantener la identidad de la misma persona a lo largo del video. `persist=True` conserva el estado del rastreador.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

generador_tracking = modelo.track(
    source=str(ruta_video),
    device=DISPOSITIVO,
    conf=0.40,
    classes=[0],
    imgsz=480,
    vid_stride=2,
    tracker='bytetrack.yaml',
    persist=True,
    stream=True,
    save=True,
    project=str(CARPETA_RESULTADOS),
    name='tracking_video',
    exist_ok=True,
    verbose=False
)

ids_unicos = set()
registros_tracking = []
carpeta_tracking = None
fotogramas_tracking = 0

for resultado_frame in generador_tracking:
    fotogramas_tracking += 1
    carpeta_tracking = Path(resultado_frame.save_dir)
    cajas = resultado_frame.boxes

    if cajas is not None and cajas.id is not None:
        ids = cajas.id.int().cpu().tolist()
        confianzas = cajas.conf.cpu().tolist()
        ids_unicos.update(ids)
        for track_id, confianza in zip(ids, confianzas):
            registros_tracking.append({
                'fotograma_procesado': fotogramas_tracking,
                'track_id': track_id,
                'confianza': round(float(confianza), 4),
            })

    if fotogramas_tracking % 100 == 0:
        print(f'Procesados: {fotogramas_tracking} | IDs vistos: {len(ids_unicos)}')

print('Tracking terminado')
print('Fotogramas analizados:', fotogramas_tracking)
print('IDs únicos observados:', len(ids_unicos))
print('Carpeta de salida:', carpeta_tracking.resolve())

In [ ]:
df_tracking = pd.DataFrame(registros_tracking)

if df_tracking.empty:
    print('No se obtuvieron IDs de tracking con el umbral actual.')
else:
    resumen_tracking = (
        df_tracking.groupby('track_id')
        .agg(
            apariciones=('track_id', 'size'),
            primer_fotograma=('fotograma_procesado', 'min'),
            ultimo_fotograma=('fotograma_procesado', 'max'),
            confianza_promedio=('confianza', 'mean'),
        )
        .sort_values('apariciones', ascending=False)
        .reset_index()
    )
    resumen_tracking['confianza_promedio'] = (resumen_tracking['confianza_promedio'] * 100).round(2)
    display(resumen_tracking.head(20))

## 10. Archivos generados y reproducción

La reproducción integrada depende del códec disponible. Si VS Code no reproduce un `.avi`, abre el archivo directamente desde el explorador o conviértelo a MP4 con la celda opcional siguiente.

In [ ]:
archivos_tracking = sorted(p for p in carpeta_tracking.glob('*') if p.is_file())
for archivo in archivos_tracking:
    print(f'{archivo.name}: {archivo.stat().st_size / 1024**2:.2f} MB')

In [ ]:
# Conversión opcional del primer video generado a MP4 para reproducirlo en el notebook.
# Requiere que OpenCV tenga habilitado el códec mp4v.
video_origen = next((p for p in archivos_tracking if p.suffix.lower() in {'.avi', '.mp4'}), None)

if video_origen is None:
    print('No se encontró un video de tracking.')
elif video_origen.suffix.lower() == '.mp4':
    display(Video(str(video_origen), embed=False))
else:
    video_mp4 = video_origen.with_suffix('.mp4')
    entrada = cv2.VideoCapture(str(video_origen))
    fps_salida = entrada.get(cv2.CAP_PROP_FPS) or 15
    ancho_salida = int(entrada.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto_salida = int(entrada.get(cv2.CAP_PROP_FRAME_HEIGHT))
    salida = cv2.VideoWriter(
        str(video_mp4),
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps_salida,
        (ancho_salida, alto_salida),
    )

    if not salida.isOpened():
        entrada.release()
        raise RuntimeError('OpenCV no pudo crear el MP4; abre el AVI directamente.')

    while True:
        ok, frame = entrada.read()
        if not ok:
            break
        salida.write(frame)

    entrada.release()
    salida.release()
    print('MP4 creado:', video_mp4.resolve())
    display(Video(str(video_mp4), embed=False))

## Conclusiones

- La detección identifica objetos de forma independiente en cada fotograma.
- El tracking intenta mantener un ID para cada objeto a través del tiempo.
- El número de IDs únicos observados no equivale necesariamente al número real de personas: una oclusión puede fragmentar una trayectoria y crear un ID nuevo.
- `stream=True`, `imgsz=480` y `vid_stride=2` reducen el uso de RAM y GPU.
- Para conteos reales, el siguiente paso es definir una línea o región de interés y contar únicamente los cruces confirmados.